# Part 17: Schema Migrations

As an ATProto PDS evolves, its database schema must change safely.
Migrations are versioned steps that transform the schema without losing data.

**What you'll learn:**
- Schema versioning
- Ordered migration execution
- Migration idempotency
- Rollback patterns

**Estimated Time:** 12-15 minutes

## Step 4: Migration Manager

A migration manager tracks the schema version and runs pending migrations
in order. Each migration is a block that modifies the schema.

In [ ]:
@interface MigrationManager : NSObject
@property (nonatomic, assign) int schemaVersion;
@property (nonatomic, strong) NSMutableDictionary *migrations;
@property (nonatomic, strong) NSMutableArray *migrationLog;
- (void)registerMigration:(NSString *)name version:(int)version;
- (int)runMigrations;
- (int)currentVersion;
@end

@implementation MigrationManager
- (instancetype)init {
    self = [super init];
    if (self) {
        _schemaVersion = 0;
        _migrations = [NSMutableDictionary dictionary];
        _migrationLog = [NSMutableArray array];
    }
    return self;
}
- (void)registerMigration:(NSString *)name version:(int)version {
    [_migrations setObject:name forKey:[NSString stringWithFormat:@"%d", version]];
    NSLog(@"Registered migration V%d: %@", version, name);
}
- (int)runMigrations {
    int runCount = 0;
    for (int v = _schemaVersion + 1; v <= 10; v++) {
        NSString *key = [NSString stringWithFormat:@"%d", v];
        NSString *name = [_migrations objectForKey:key];
        if (name == nil) break;
        NSLog(@"Running V%d: %@", v, name);
        [_migrationLog addObject:[NSString stringWithFormat:@"V%d: %@", v, name]];
        _schemaVersion = v;
        runCount = runCount + 1;
    }
    NSLog(@"Ran %d migrations, now at V%d", runCount, _schemaVersion);
    return runCount;
}
- (int)currentVersion {
    return _schemaVersion;
}
@end

## Step 5: Full Blob + Migration Demo

Upload blobs, verify CID integrity. Then register and run migrations.

In [ ]:
// Blob support classes (from atproto-blobs notebook)
@interface CIDHelper : NSObject
- (NSString *)cidForData:(NSData *)data;
@end

@implementation CIDHelper
- (NSString *)cidForData:(NSData *)data {
    int a = 0, b = 0, c = 0, d = 0;
    const char *bytes = [data bytes];
    int len = [data length];
    for (int i = 0; i < len; i++) {
        a = a ^ bytes[i];
        b = b ^ (bytes[i] << 1);
        c = c ^ (bytes[i] << 2);
        d = d ^ (bytes[i] << 3);
    }
    return [NSString stringWithFormat:@"bafyrei%02x%02x%02x%02x", a & 0xFF, b & 0xFF, c & 0xFF, d & 0xFF];
}
@end

@interface BlobStore : NSObject
@property (nonatomic, strong) NSMutableDictionary *blobs;
@property (nonatomic, strong) NSMutableDictionary *metadata;
@property (nonatomic, strong) CIDHelper *cidHelper;
- (NSDictionary *)uploadBlob:(NSData *)data mimeType:(NSString *)mimeType forDid:(NSString *)did;
- (NSData *)getBlob:(NSString *)cid;
@end

@implementation BlobStore
- (instancetype)init {
    self = [super init];
    if (self) {
        _blobs = [NSMutableDictionary dictionary];
        _metadata = [NSMutableDictionary dictionary];
        _cidHelper = [[CIDHelper alloc] init];
    }
    return self;
}
- (NSDictionary *)uploadBlob:(NSData *)data mimeType:(NSString *)mimeType forDid:(NSString *)did {
    NSString *cid = [_cidHelper cidForData:data];
    if ([_blobs objectForKey:cid] != nil) {
        NSLog(@"Blob already exists: %@ (dedup)", cid);
        NSMutableDictionary *meta = [NSMutableDictionary dictionaryWithDictionary:[_metadata objectForKey:cid]];
        int refs = [[meta objectForKey:@"refs"] intValue] + 1;
        [meta setObject:[NSString stringWithFormat:@"%d", refs] forKey:@"refs"];
        [_metadata setObject:meta forKey:cid];
        return meta;
    }
    [_blobs setObject:data forKey:cid];
    NSDictionary *meta = @{
        @"cid": cid,
        @"mimeType": mimeType,
        @"size": [NSString stringWithFormat:@"%d", [data length]],
        @"did": did,
        @"refs": @"1"
    };
    [_metadata setObject:meta forKey:cid];
    NSLog(@"Uploaded blob: %@ (%d bytes, %@)", cid, [data length], mimeType);
    return meta;
}
- (NSData *)getBlob:(NSString *)cid {
    return [_blobs objectForKey:cid];
}
@end

// Blob operations
BlobStore *blobStore = [[BlobStore alloc] init];
NSData *avatar = [NSData dataWithBytes:"avatar" length:6];
NSDictionary *avatarMeta = [blobStore uploadBlob:avatar mimeType:@"image/png" forDid:@"did:plc:alice"];
NSLog(@"Avatar CID: %@", [avatarMeta objectForKey:@"cid"]);

// Migration operations
MigrationManager *mm = [[MigrationManager alloc] init];
[mm registerMigration:@"Create accounts table" version:1];
[mm registerMigration:@"Add invite codes table" version:2];
[mm registerMigration:@"Add blobs table" version:3];
[mm registerMigration:@"Add moderation_subjects table" version:4];

NSLog(@"Before: V%d", [mm currentVersion]);
[mm runMigrations];
NSLog(@"After: V%d", [mm currentVersion]);

// Running again does nothing (already at V4)
int rerun = [mm runMigrations];
NSLog(@"Re-ran %d migrations", rerun);

## Exercise

Add `rollbackMigration:` to `MigrationManager` that decrements the schema version by 1.
This is useful for testing and represents how a PDS might handle a failed migration.

Hint: subtract 1 from schemaVersion and log the rollback.

In [ ]:
// Exercise: implement rollbackMigration:
// Your code here...

## Solution

In [ ]:
// Solution: rollback decrements version
@interface MigrationManager2 : NSObject
@property (nonatomic, assign) int schemaVersion;
- (void)rollbackMigration;
@end

@implementation MigrationManager2
- (void)rollbackMigration {
    if (_schemaVersion > 0) {
        _schemaVersion = _schemaVersion - 1;
        NSLog(@"Rolled back to V%d", _schemaVersion);
    }
}
@end

// Test it
MigrationManager2 *mm2 = [[MigrationManager2 alloc] init];
mm2.schemaVersion = 3;
NSLog(@"Before: V%d", mm2.schemaVersion);
[mm2 rollbackMigration];
NSLog(@"After: V%d", mm2.schemaVersion);
[mm2 rollbackMigration];
NSLog(@"Rolled back again: V%d", mm2.schemaVersion);

## Next Steps

Migrations keep a PDS schema in sync with the protocol as it evolves. Next, explore the full lifecycle by building account services and XRPC endpoints.

## What to Read Next

You now understand schema migrations. Next:
- **Part 18: Archaeology — Repository** — how production MST/CAR/CBOR code works
- **Part 19: Archaeology — PDS Services** — how production services compose